In [3]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

# Load the dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.boston_housing.load_data()

print(f"Training samples: {x_train.shape}")
print(f"Test samples: {x_test.shape}")
print(f"Number of features: {x_train.shape[1]}")

57026/57026 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Training samples: (404, 13)
Test samples: (102, 13)
Number of features: 13


In [5]:
# Normalizing features using train set mean and std
mean = x_train.mean(axis=0)
std = x_train.std(axis=0)

x_train = (x_train - mean) / std
x_test = (x_test - mean) / std

In [10]:

keras.backend.clear_session()

# Building a baseline model with hardcoded hyperparameters
b_model = keras.Sequential([
    keras.Input(shape=(x_train.shape[1],)),
    keras.layers.Dense(units=128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(1)
])

b_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 128)                 │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 10,113 (39.50 KB)

 Trainable params: 10,113 (39.50 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
NUM_EPOCHS = 100

b_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.MeanSquaredError(),
    metrics=['mae']
)

b_history = b_model.fit(
    x_train, y_train,
    epochs=NUM_EPOCHS,
    validation_split=0.2,
    verbose=0  # Suppresses per-epoch output to keep things clean
)

print("Baseline training complete")

Baseline training complete


In [13]:
b_eval_dict = b_model.evaluate(x_test, y_test, return_dict=True)
print(f"\nBaseline Model Results:")
print(f"Units in Dense layer: {b_model.get_layer('dense_1').units}")
print(f"Learning rate: {b_model.optimizer.learning_rate.numpy()}")
print(f"Test Loss (MSE): {b_eval_dict['loss']:.4f}")
print(f"Test MAE: {b_eval_dict['mae']:.4f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 22.6257 - mae: 2.9741

Baseline Model Results:
Units in Dense layer: 64
Learning rate: 0.0010000000474974513
Test Loss (MSE): 22.6257
Test MAE: 2.9741


In [14]:
def model_builder(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(x_train.shape[1],)))

    # Tune units in first Dense layer: 32 to 256
    hp_units = hp.Int('units', min_value=32, max_value=256, step=32)
    model.add(keras.layers.Dense(units=hp_units, activation='relu'))
    model.add(keras.layers.Dropout(0.2))
    model.add(keras.layers.Dense(64, activation='relu'))
    model.add(keras.layers.Dense(1))

    # Tune learning rate
    hp_lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp_lr),
        loss=keras.losses.MeanSquaredError(),
        metrics=['mae']
    )

    return model

In [15]:
tuner = kt.RandomSearch(
    model_builder,
    objective='val_mae',
    max_trials=10,
    directory='kt_dir',
    project_name='kt_randomsearch',
    overwrite=True
)

tuner.search_space_summary()

Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 256, 'step': 32, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}


In [17]:
stop_early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

tuner.search(
    x_train, y_train,
    epochs=NUM_EPOCHS,
    validation_split=0.2,
    callbacks=[stop_early],
    verbose=0
)

print("Hyperparameter search complete")

Hyperparameter search complete


In [18]:
best_hps = tuner.get_best_hyperparameters()[0]

print(f"""
Hyperparameter search complete!
Optimal units in first Dense layer: {best_hps.get('units')}
Optimal learning rate: {best_hps.get('learning_rate')}
""")


Hyperparameter search complete!
Optimal units in first Dense layer: 96
Optimal learning rate: 0.01



In [19]:
h_model = tuner.hypermodel.build(best_hps)
h_model.summary()

h_history = h_model.fit(
    x_train, y_train,
    epochs=NUM_EPOCHS,
    validation_split=0.2,
    verbose=0
)

print("Hypertuned model training complete")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                      │ (None, 96)                  │           1,344 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 96)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 64)                  │           6,208 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,617 (29.75 KB)

 Trainable params: 7,617 (29.75 KB)

 Non-trainable params: 0 (0.00 B)

Hypertuned model training complete


In [20]:
h_eval_dict = h_model.evaluate(x_test, y_test, return_dict=True)

print("\n RESULTS COMPARISON")
print(f"\nBASELINE MODEL:")
print(f"  Units in 1st Dense layer : 128")
print(f"  Learning rate            : 0.001")
print(f"  Test Loss (MSE)          : {b_eval_dict['loss']:.4f}")
print(f"  Test MAE                 : {b_eval_dict['mae']:.4f}")

print(f"\nHYPERTUNED MODEL:")
print(f"  Units in 1st Dense layer : {best_hps.get('units')}")
print(f"  Learning rate            : {best_hps.get('learning_rate')}")
print(f"  Test Loss (MSE)          : {h_eval_dict['loss']:.4f}")
print(f"  Test MAE                 : {h_eval_dict['mae']:.4f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 24.2600 - mae: 3.2451

 RESULTS COMPARISON

BASELINE MODEL:
  Units in 1st Dense layer : 128
  Learning rate            : 0.001
  Test Loss (MSE)          : 22.6257
  Test MAE                 : 2.9741

HYPERTUNED MODEL:
  Units in 1st Dense layer : 96
  Learning rate            : 0.01
  Test Loss (MSE)          : 24.2600
  Test MAE                 : 3.2451


## Conclusion

### Model Architecture
The model used in this lab is a fully-connected (Dense) neural network. It consists of:
- **Input layer**: accepts the 13 numerical features of the Boston Housing dataset
- **First Dense layer**: the main tunable layer which learns complex patterns from the input features. More units means more capacity to learn, but also more risk of overfitting.
- **Dropout layer (0.2)**: randomly disables 20% of neurons during training to prevent overfitting
- **Second Dense layer**: further refines the learned representations
- **Output layer**: a single neuron with no activation, outputting a continuous price prediction )

### Hyperparameters Tuned & Why
Two hyperparameters were selected for tuning:
- **Units in the first Dense layer**: directly controls the model's learning capacity. Too few units underfit the data; too many overfit it. We searched between 32 and 256 in steps of 32.
- **Learning rate**: controls how aggressively the optimizer updates weights. A rate too high causes overshooting; too low causes slow or stuck training. We tested 0.01, 0.001, and 0.0001.

### RandomSearch Tuner
Instead of Hyperband, we used **RandomSearch** which is a tuner that randomly samples hyperparameter combinations from the defined search space and evaluates each one. It is simple, effective, and avoids any bias in the search direction. With `max_trials=10`, it tested 10 randomly chosen combinations out of 21 possible ones.

### Results & Observation
Interestingly, the baseline slightly outperformed the hypertuned model (MAE: 2.97 vs 3.24). 
This is a known limitation of automated tuning on small datasets with only 404 training 
samples, 10 random trials are not enough to reliably identify a better configuration. 
To validate this, we repeat the same process on Fashion MNIST (60,000 samples) in the 
next section, where we expect hyperparameter tuning to show a clearer improvement.

In [21]:
# Loading Fashion MNIST dataset
(img_train, label_train), (img_test, label_test) = keras.datasets.fashion_mnist.load_data()

# Normalizing pixel values
img_train = img_train.astype('float32') / 255.0
img_test = img_test.astype('float32') / 255.0

print(f"Training samples: {img_train.shape}")
print(f"Test samples: {img_test.shape}")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Training samples: (60000, 28, 28)
Test samples: (10000, 28, 28)


In [22]:
keras.backend.clear_session()

fm_b_model = keras.Sequential([
    keras.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(10, activation='softmax')
])

fm_b_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

fm_b_history = fm_b_model.fit(
    img_train, label_train,
    epochs=10,
    validation_split=0.2,
    verbose=0
)

fm_b_eval = fm_b_model.evaluate(img_test, label_test, return_dict=True)
print(f"Baseline Accuracy: {fm_b_eval['accuracy']:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8798 - loss: 0.3620   
Baseline Accuracy: 0.8798


In [23]:
def fm_model_builder(hp):
    model = keras.Sequential([
        keras.Input(shape=(28, 28)),
        keras.layers.Flatten()
    ])

    hp_units = hp.Int('units', min_value=32, max_value=512, step=32)
    model.add(keras.layers.Dense(units=hp_units, activation='relu'))
    model.add(keras.layers.Dropout(0.2))
    model.add(keras.layers.Dense(10, activation='softmax'))

    hp_lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp_lr),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    return model

fm_tuner = kt.RandomSearch(
    fm_model_builder,
    objective='val_accuracy',
    max_trials=10,
    directory='kt_dir',
    project_name='kt_fashionmnist',
    overwrite=True
)

stop_early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

fm_tuner.search(
    img_train, label_train,
    epochs=10,
    validation_split=0.2,
    callbacks=[stop_early],
    verbose=0
)

print("Search complete")

Search complete


In [24]:
# Get best hyperparameters
fm_best_hps = fm_tuner.get_best_hyperparameters()[0]
print(f"Optimal units: {fm_best_hps.get('units')}")
print(f"Optimal learning rate: {fm_best_hps.get('learning_rate')}")

# Build and train the hypertuned model
fm_h_model = fm_tuner.hypermodel.build(fm_best_hps)

fm_h_model.fit(
    img_train, label_train,
    epochs=10,
    validation_split=0.2,
    verbose=0
)

# Evaluate
fm_h_eval = fm_h_model.evaluate(img_test, label_test, return_dict=True)

# Final comparison
print("\n FASHION MNIST RESULTS COMPARISON")
print(f"\nBASELINE MODEL:")
print(f"  Units in 1st Dense layer : 512")
print(f"  Learning rate : 0.001")
print(f"  Test Accuracy : {fm_b_eval['accuracy']:.4f}")

print(f"\nHYPERTUNED MODEL:")
print(f"  Units in 1st Dense layer : {fm_best_hps.get('units')}")
print(f"  Learning rate : {fm_best_hps.get('learning_rate')}")
print(f"  Test Accuracy : {fm_h_eval['accuracy']:.4f}")

Optimal units: 256
Optimal learning rate: 0.001
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8800 - loss: 0.3406   

 FASHION MNIST RESULTS COMPARISON

BASELINE MODEL:
  Units in 1st Dense layer : 512
  Learning rate : 0.001
  Test Accuracy : 0.8798

HYPERTUNED MODEL:
  Units in 1st Dense layer : 256
  Learning rate : 0.001
  Test Accuracy : 0.8800


## Validation: Fashion MNIST Results

On the larger Fashion MNIST dataset (60,000 samples), the hypertuned model achieved 88.00% accuracy compared to the baseline's 87.98%. While the accuracy gain is marginal, the more meaningful finding is efficiency the tuner identified that 256 units performs equivalently to the baseline's hardcoded 512 units, producing a leaner model that uses half the parameters with no accuracy cost.

This contrasts with the Boston Housing experiment, where tuning actually hurt performance. The difference comes down to dataset size with only 404 samples, 10 random trials are not enough for the tuner to reliably distinguish good hyperparameter combinations from bad ones. With 60,000 samples, each trial gets sufficient data to be meaningfully evaluated, and the tuner's recommendations become trustworthy.

The overall conclusion is that hyperparameter tuning is not a guaranteed improvement its effectiveness scales with the size and richness of the dataset. On small datasets, a well-reasoned manual choice can outperform automated search. On larger datasets, automated tuning reliably finds more efficient configurations even when raw accuracy gains are small.